# HAM10000 Exploratory Data Analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

DATA_DIR = Path("../data/raw")
META_PATH = DATA_DIR / "HAM10000_metadata.csv"

In [ ]:
CLASS_NAMES = {
    "nv": "melanocytic nevi",
    "mel": "melanoma",
    "bkl": "benign keratosis",
    "bcc": "basal cell carcinoma",
    "akiec": "actinic keratoses",
    "vasc": "vascular lesion",
    "df": "dermatofibroma",
}
MALIGNANT = {"mel", "bcc", "akiec"}

## Load metadata

In [ ]:
df = pd.read_csv(META_PATH)
assert set(df["dx"].unique()) <= set(CLASS_NAMES), "Unexpected class label found"
print(f"Rows: {len(df)}, unique lesions: {df['lesion_id'].nunique()}")
df.head()

## Class balance

In [ ]:
counts = df["dx"].value_counts()
pct = (counts / len(df) * 100).round(1)

for cls in counts.index:
    flag = " (MALIGNANT)" if cls in MALIGNANT else ""
    print(f"{cls:6s} {CLASS_NAMES[cls]:25s} n={counts[cls]:5d} {pct[cls]:5.1f}%{flag}")

In [ ]:
lesion_level = df.drop_duplicates("lesion_id")
lcounts = lesion_level["dx"].value_counts()
lcounts

## Lesion duplication

In [ ]:
images_per_lesion = df.groupby("lesion_id").size()
multi = (images_per_lesion > 1).sum()

print(f"Unique lesions: {df['lesion_id'].nunique()}")
print(f"Lesions with >1 image: {multi} ({multi / df['lesion_id'].nunique() * 100:.1f}%)")
print(f"Max images for a single lesion: {images_per_lesion.max()}")

## Diagnostic confirmation method

In [ ]:
df["dx_type"].value_counts()

## Missing values

In [ ]:
print(df.isna().sum())
print(f"\nsex == 'unknown': {(df['sex'] == 'unknown').sum()}")

## Image properties

In [ ]:
part1 = DATA_DIR / "HAM10000_images_part_1"
part2 = DATA_DIR / "HAM10000_images_part_2"

sample_ids = df["image_id"].sample(30, random_state=42)
sizes, modes = set(), set()

for img_id in sample_ids:
    path = part1 / f"{img_id}.jpg"
    if not path.exists():
        path = part2 / f"{img_id}.jpg"
    with Image.open(path) as im:
        sizes.add(im.size)
        modes.add(im.mode)

print("Distinct sizes:", sizes)
print("Distinct color modes:", modes)

## Visual sample: one image per class

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, cls in zip(axes, CLASS_NAMES):
    row = df[df["dx"] == cls].sample(1, random_state=1).iloc[0]
    path = part1 / f"{row['image_id']}.jpg"
    if not path.exists():
        path = part2 / f"{row['image_id']}.jpg"
    img = Image.open(path)
    ax.imshow(img)
    ax.set_title(f"{cls} - {CLASS_NAMES[cls]}", fontsize=10)
    ax.axis("off")

axes[-1].axis("off")
plt.tight_layout()
plt.show()

## Body site (localization) by class

In [ ]:
loc_by_class = pd.crosstab(df["dx"], df["localization"])
loc_by_class

In [ ]:
loc_pct = pd.crosstab(df["dx"], df["localization"], normalize="index") * 100

fig, ax = plt.subplots(figsize=(14, 6))
loc_pct.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
ax.set_ylabel("% of class")
ax.set_title("Body site distribution within each diagnosis class")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## Age distribution by class

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
df.boxplot(column="age", by="dx", ax=ax)
ax.set_xlabel("Diagnosis")
ax.set_ylabel("Age")
ax.set_title("Age distribution by diagnosis class")
plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
df.groupby("dx")["age"].describe()